# 3DGS Pipeline 控制中枢

**使用方法**：
1. 运行 **Cell 1（初始化）**，加载所有函数和配置
2. 按需运行各 Section 中的单元格
3. 修改参数？编辑 ，重新运行 Cell 1 即可

> 每个 Section 均可独立运行，无需依赖上方单元格的执行状态。

In [3]:
# ═══════════════════════════════════════════════════════
# ① 初始化（每次打开 Notebook 只需运行这一个 Cell）
# ═══════════════════════════════════════════════════════
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

os.environ["CUDA_DEVICE_ORDER"]        = "PCI_BUS_ID"          # 让 CUDA 编号与 nvidia-smi 一致
os.environ["CUDA_VISIBLE_DEVICES"]     = "0"                    # 0 = RTX 2080 Ti
os.environ["PYTORCH_CUDA_ALLOC_CONF"]  = "expandable_segments:False"

# 确保 src/ 在 Python 路径中（支持从任意目录打开 notebook）
_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / "src" / "pipeline" / "__init__.py").exists()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.pipeline import *

cfg = load_config()   # 读取 configs/pipeline.yaml
print(f"✓ 项目根目录 : {PROJECT_ROOT}")
print(f"✓ 数据集     : {cfg['dataset']['source']}  →  {cfg['dataset']['path']}")
print(f"✓ 训练输出   : {cfg['training']['output_dir']}")
print(f"✓ 迭代次数   : {cfg['training']['iterations']}")
print(f"✓ 查看器     : {cfg['viewer']['backend']}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ 项目根目录 : /home/ansatz/github/ME6402-3D-Autonomous-Retail
✓ 数据集     : custom  →  /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/team_data2
✓ 训练输出   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter
✓ 迭代次数   : 30000
✓ 查看器     : sibr


## Section 1 — 环境检查

In [4]:
# 检查 PyTorch、CUDA、COLMAP、3DGS CUDA 模块、Open3D、Docker
check_environment(cfg)


3DGS 环境检查

PyTorch:  2.1.2
CUDA 可用: True
CUDA 版本: 11.8
  GPU 0: NVIDIA GeForce RTX 2080 Ti  (21.7 GB)

核心依赖:
  ✓ OpenCV  4.8.1
  ✓ NumPy  1.26.4
  ✓ plyfile
  ✓ SciPy  1.11.4
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-04-07 19:18:23,753  INFO  环境检查完成，结果: 通过


  ✓ Open3D  0.19.0
  ✓ diff_gaussian_rasterization (CUDA 模块)

COLMAP:
  ✓ /usr/bin/colmap

Docker（SIBR 查看器）:
  ✓ Docker daemon 可访问

项目目录:
  PROJECT_ROOT : /home/ansatz/github/ME6402-3D-Autonomous-Retail
  GS_DIR       : /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/gaussian-splatting  ✓
  DATA_DIR     : /home/ansatz/github/ME6402-3D-Autonomous-Retail/data  ✓
  OUTPUT_DIR   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs  ✓

✅ 环境检查通过


True

## Section 2 — 视频抽帧（可选）

适用场景：你有一段录制的视频，需要先抽帧再走 COLMAP 流程。

**配置方式**：在  中修改  块，将 ，
然后重新运行 **Cell 1**，再运行本 Cell。

In [ ]:
# 抽帧完成后会自动更新 cfg，指向新场景目录
# 完成后直接运行 Section 3 (COLMAP) 即可
extract_frames(cfg)


ℹ️  video.enabled=false，跳过抽帧。
   若要抽帧，请在 configs/pipeline.yaml 中将 video.enabled 改为 true。


False

## Section 3 — COLMAP 相机标定

适用场景：自有数据（视频抽帧或自拍照片），需要从图像推导相机参数。
官方数据集（T&T、DB）已自带相机参数，**无需此步骤**。

**配置方式**：在  中将 ，
并确认  和  正确。

In [4]:
# COLMAP 五步流程：特征提取 → 匹配 → 稀疏重建 → 畸变校正 → 内参回填
# 完成后 cfg["dataset"]["path"] 自动切换到 undistorted dense 输出
run_colmap(cfg)



🧭 运行 COLMAP  →  /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2
   bash /home/ansatz/github/ME6402-3D-Autonomous-Retail/scripts/reconstruction/run_colmap.sh /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/team_data2/images /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2

Feature extraction

Processed file [1/117]
  Name:            IMG_20260402_120500.jpg
  SKIP: Features for image already extracted.
Processed file [2/117]
  Name:            IMG_20260402_120505.jpg
  SKIP: Features for image already extracted.
Processed file [3/117]
  Name:            IMG_20260402_120508.jpg
  SKIP: Features for image already extracted.
Processed file [4/117]
  Name:            IMG_20260402_120512.jpg
  SKIP: Features for image already extracted.
Processed file [5/117]
  Name:            IMG_20260402_120516.jpg
  SKIP: Features for image already extracted.
Processed file [6/117]
  Name:            IMG_20260402_120523.jpg
  SKIP: Fe

KeyboardInterrupt: 

## Section 4 — 3DGS 训练

关键参数（在  →  块修改）：
- ：迭代次数（300~5000 快速验证；30000 高质量）
- ：分辨率倍率（1=原始；2=1/2；RTX 4060 建议从 2 开始）
- ：输出根目录（每次训练自动创建子目录）

In [6]:
# OOM 时自动降档重试（resolution ×1 → ×2 → ×4）
run_training(cfg)


2026-04-07 19:24:40,678  INFO  ==================================================
2026-04-07 19:24:40,679  INFO  启动 3DGS 训练
2026-04-07 19:24:40,679  INFO  ==================================================
2026-04-07 19:24:40,680  INFO  训练前自动修正 dataset.path: /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense
2026-04-07 19:24:40,680  INFO  训练命令: python train.py -s /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense -m /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter --iterations 30000 --resolution 2 --sh_degree 3 --save_iterations 7000 30000 --test_iterations 7000 30000


🧭 训练前自动修正数据路径 →  /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense

⏳ 训练开始
   数据   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense
   迭代   : 30000
   输出   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter
   分辨率 : ×2
Optimizing /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter
Output folder: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter [07/04 19:24:41]
Tensorboard not available: not logging progress [07/04 19:24:41]

Reading camera 1/117
Reading camera 2/117
Reading camera 3/117
Reading camera 4/117
Reading camera 5/117
Reading camera 6/117
Reading camera 7/117
Reading camera 8/117
Reading camera 9/117
Reading camera 10/117
Reading camera 11/117
Reading camera 12/117
Reading camera 13/117
Reading camera 14/117
Reading camera 15/117
Reading camera 16/117
Reading camera 17/117
Reading camera 18/117
Reading camera 19/117
Read

KeyboardInterrupt: 

## Section 5 — 查看训练结果

自动搜索  下最新的 ，用 Open3D 打开交互窗口。

如需查看 SIBR，在  中将 ，
重新运行 Cell 1 后再运行此 Section。

In [ ]:
# Open3D 交互查看（关闭窗口后继续）
# 也可传入指定路径：open_viewer(cfg, ply_path="outputs/xxx/point_cloud/iteration_300/point_cloud.ply")
open_viewer(cfg)


✅ 找到点云：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter/point_cloud/iteration_7000/point_cloud.ply
   大小：75.7 MB
   点数：320,263
   打开 Open3D 交互窗口（关闭窗口后继续）...


2026-04-05 01:47:38,939  INFO  Open3D 查看完成: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/team_data2_30000iter/point_cloud/iteration_7000/point_cloud.ply


True

In [ ]:
# 分析训练结果：列出 PLY 文件、打印 results.json
analyze_results(cfg)


## Section 6 — SIBR Viewer（可选）

如需在 SIBR 中查看，运行此 Cell。会列出可用模型供你选择。

**前提**：已构建 Docker 镜像（见 [info] Building image: sibr-builder:ubuntu22.04-cuda11.8
Sending build context to Docker daemon  12.01GB

Step 1/4 : FROM nvidia/cuda:11.8.0-devel-ubuntu22.04
 ---> 6f9cc9f1ba9e
Step 2/4 : ENV DEBIAN_FRONTEND=noninteractive
 ---> Using cache
 ---> db877d8112e1
Step 3/4 : RUN apt-get update && apt-get install -y --no-install-recommends     build-essential     cmake     ninja-build     git     pkg-config     libglew-dev     libassimp-dev     libboost-all-dev     libgtk-3-dev     libopencv-dev     libglfw3-dev     libavdevice-dev     libavcodec-dev     libavformat-dev     libswscale-dev     libeigen3-dev     libxxf86vm-dev     libembree-dev     libgl1-mesa-dev     libglu1-mesa-dev     libx11-dev     libxext-dev     libxrender-dev     libxrandr-dev     libxinerama-dev     libxcursor-dev     ca-certificates     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> feb68a7c28aa
Step 4/4 : WORKDIR /workspace
 ---> Using cache
 ---> 8fdbd6c2ecd5
Successfully built 8fdbd6c2ecd5
Successfully tagged sibr-builder:ubuntu22.04-cuda11.8
[done] Image built: sibr-builder:ubuntu22.04-cuda11.8）

In [19]:
# 交互式选择模型并在 Docker 内启动 SIBR
launch_sibr(cfg)



可用模型（按最新迭代降序）:
  [1] team_data2_30000iter  (iter=30000)
  [2] 7dfdb283-b  (iter=30000)
  [3] 3dgs_team_data2_30000iter  (iter=30000)
  [4] 3dgs_tandt_30000iter  (iter=30000)
  [5] 3dgs_tandt_5000iter  (iter=5000)
  [6] 3dgs_custom_scene_01_5000iter  (iter=5000)
  [7] 3dgs_demo_300iter  (iter=300)
  [8] smoke_truck_10iter  (iter=10)
  [9] 3dgs_demo  (iter=?)

🖼️  启动 SIBR Viewer ...
   bash /home/ansatz/github/ME6402-3D-Autonomous-Retail/scripts/reconstruction/run_sibr_in_docker.sh sibr-builder:ubuntu22.04-cuda11.8 /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   提示：关闭 SIBR 窗口后，该单元继续运行。
[info] Launching SIBR viewer in container...

== CUDA ==

CUDA Version 11.8.0

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://d

[SIBR] ##  ERROR  ##:	FILE /workspace/third_party/gaussian-splatting/SIBR_viewers/src/core/scene/ParseData.cpp
			LINE 560, FUNC getParsedData
			Cannot determine type of dataset at //root/gpufree-data/gaussian-splatting/image


[SIBR] --  INFOS  --:	Initializing Raycaster
[SIBR] --  INFOS  --:	Interactive camera using (0.009,1100) near/far planes.
Switched to trackball mode.


2026-04-07 19:48:17,987  INFO  SIBR 查看完成: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter，ok=True
INFO:pipeline:SIBR 查看完成: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter，ok=True


[SIBR] --  INFOS  --:	Deinitialization of GLFW
[done] SIBR viewer exited.
✓ SIBR 正常退出


True

## Section 7 — 一键完整流程

参数调好后，一口气执行：COLMAP（可选）→ 训练 → 查看结果。

In [ ]:
# 一键流程（是否跑 COLMAP 取决于 cfg["dataset"]["use_colmap"]）
run_pipeline(cfg)


## Section 9 — SAGA 3D 语义分割

**Segment Any 3D Gaussians**（AAAI 2025）：在已训练好的 3DGS 模型上直接添加语义特征，无需重新训练重建，无需标注数据。

**流程概览**
1. 安装 SAGA 依赖（只需做一次）
2. 下载 SAM ViT-H checkpoint（~2.5 GB，只需做一次）
3. 提取 SAM 特征 + mask（按场景做一次）
4. 训练对比特征（~10-40 分钟）
5. 打开 SAGA Notebook → 文字/点击 → 输出 3D Bounding Box

**配置方式**：在 `configs/pipeline.yaml` 的 `saga:` 块中设置 `image_root` 和 `model_path`，然后重新运行 **Cell 1**。

### 9.1  导入 SAGA 模块 + 加载配置


In [9]:
from src.pipeline.saga import *

saga_cfg = load_saga_config(cfg)   # 读取 pipeline.yaml 中的 saga: 块

print(f"模型路径  : {saga_cfg['model_path']}")
print(f"场景图像  : {saga_cfg['image_root']}")
print(f"SAM ckpt  : {saga_cfg['sam_checkpoint']}")
print(f"降采样    : ×{saga_cfg['downsample']}")


模型路径  : outputs/3dgs_team_data2_30000iter
场景图像  : data/colmap_workspace/team_data2/dense
SAM ckpt  : dependencies/sam_ckpt/sam_vit_h_4b8939.pth
降采样    : ×4


### 9.2  SAGA 环境检查（确认依赖全部就绪）

In [10]:
check_saga_ready()


SAGA 环境检查
  ✓ SAGA 仓库  (/home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA)
  ✓ 子模块 diff-gaussian-rasterization_contrastive_f
  ✓ segment-anything (SAM)
  ✓ kmeans_pytorch
  ✓ open_clip_torch
  ✓ hdbscan
  ✓ SAM ViT-H checkpoint  (sam_vit_h_4b8939.pth)

✅ SAGA 环境就绪


True

In [11]:
# 9.3  首次使用：安装 SAGA 依赖
# 若 9.2 显示全部 ✓，可跳过本单元格
import subprocess, sys
result = subprocess.run(
    ["bash", "scripts/setup_saga.sh"],
    cwd=str(PROJECT_ROOT)
)
print("✅ 安装完成" if result.returncode == 0 else "✗ 安装失败，查看上方输出")


 SAGA 安装脚本
 项目根目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail
✓ Conda 环境：gaussian_splatting
✓ SAGA 仓库已存在，更新子模块...

>>> 安装 segment-anything...
✓ segment-anything 安装完成（本地源）

>>> 安装 kmeans_pytorch...
✓ kmeans_pytorch 安装完成（本地源）

>>> 安装 open_clip_torch, hdbscan...
✓ open_clip_torch, hdbscan 安装完成

>>> 编译 diff-gaussian-rasterization_contrastive_f...


ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# 9.4  下载 SAM ViT-H checkpoint（~2.5 GB，只需一次）
# 若 dependencies/sam_ckpt/sam_vit_h_4b8939.pth 已存在可跳过
download_sam_checkpoint()


✓ SAM checkpoint 已存在：/home/ansatz/github/ME6402-3D-Autonomous-Retail/dependencies/sam_ckpt/sam_vit_h_4b8939.pth


True

### 9.5 — 指定要分割的场景和模型

修改下方两个变量（或直接在 `configs/pipeline.yaml` 的 `saga:` 块中修改后重新运行 Cell 1）：

```
saga:
  image_root: data/official/tandt_db/db/playroom   # 场景图像目录（含 images/ 子目录）
  model_path: outputs/playroom_30000iter            # 训练好的 3DGS 模型目录
```

> ⚠️ Teammate 的模型（`Teamate_output_unzipped/...`）训练于 Google Colab，本地没有对应图像，需使用本地训练的模型或用自己拍摄的数据。

In [ ]:
# 9.5  确认场景和模型路径（由 configs/pipeline.yaml 的 saga: 块控制）
# 如需临时覆盖，取消下方注释并修改：
# saga_cfg["image_root"] = str(PROJECT_ROOT / "data/colmap_workspace/custom_scene_01/dense")
# saga_cfg["model_path"] = str(PROJECT_ROOT / "outputs/3dgs_custom_scene_01_5000iter")

print(f"模型路径  : {saga_cfg['model_path']}")
print(f"场景图像  : {saga_cfg['image_root']}")
print(f"SAM ckpt  : {saga_cfg['sam_checkpoint']}")
print(f"降采样    : ×{saga_cfg['downsample']}")


模型路径  : outputs/3dgs_team_data2_30000iter
场景图像  : data/colmap_workspace/team_data2/dense
SAM ckpt  : dependencies/sam_ckpt/sam_vit_h_4b8939.pth
降采样    : ×1


### 9.6 — 提取特征 + 训练（按顺序 9.6.1 → 9.6.2 → 9.6.3 → 9.6.4 运行一次即可）

每个场景只需运行一次，结果会缓存在场景目录下。

| 步骤 | 单元格 | 输出目录 | 耗时 |
|------|--------|----------|------|
| SAM mask 提取 | 9.6.1 | `sam_masks/` | ~6 分钟 |
| CLIP 特征提取 | 9.6.2 | `clip_features/` | ~2 分钟 |
| 3D 尺度估算 | 9.6.3 | `mask_scales/` | ~3 分钟 |
| 对比特征训练 | 9.6.4 | `point_cloud/.../contrastive_*.ply` | 10-40 分钟 |


#### 9.6.1  Step 1：生成缩小图像 + 提取 SAM 自动分割 mask（输出到 <image_root>/sam_masks/）

In [11]:
# VRAM：~7 GB，建议在 RTX 2080 Ti 上运行
extract_sam_masks(saga_cfg)


✓ SAM mask 已存在（117 个），跳过。如需重新生成，删除 /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/sam_masks/


True

#### 9.6.2  Step 2：从图像 + SAM mask 提取 CLIP 语义特征（输出到 <image_root>/clip_features/）


In [ ]:
# VRAM：~4 GB
extract_sam_features(saga_cfg)


   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/images（117 张）

[SAGA 9.6.2] 提取 CLIP 语义特征（从 SAM mask）...
   图像目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/images
   输出目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense/clip_features
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/get_clip_features.py --image_root ...
Embedding dimension 512

0it [00:00, ?it/s]/home/ansatz/miniconda3/envs/gaussian_splatting/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future de

True

In [ ]:
from src.pipeline.saga import validate_clip_features
validate_clip_features(saga_cfg, queries=["coca cola can", "facial tissue box", "potato chips bag", "juice drink pouch"])


CLIP 特征文件统计
  文件数：117
  特征维度：torch.Size([116, 512])  (N_mask × 512)
  示例文件：IMG_20260402_120500.pt
  总 mask 数：12500
  特征范数均值：13.5826（≈1.0 表示已归一化）
Embedding dimension 512

文字查询相似度验证（top-5 分数）
  'coca cola can': top5=['0.797', '0.785', '0.783', '0.780', '0.779']  mean=0.362
  分布: |  ######## ######## ####### #     |  (0→1)
  'facial tissue box': top5=['0.821', '0.821', '0.817', '0.816', '0.807']  mean=0.388
  分布: |  ######## ######## ######## ####     |  (0→1)
  'potato chips bag': top5=['0.878', '0.875', '0.870', '0.858', '0.856']  mean=0.386
  分布: |  ######## ######## ######## #### #    |  (0→1)
  'juice drink pouch': top5=['0.806', '0.796', '0.789', '0.784', '0.780']  mean=0.325
  分布: | ## ######## ######## ###### ##     |  (0→1)

✓ 验证完成。分数 top5 越高（越接近1.0）代表 CLIP 特征识别能力越强。


In [21]:
from importlib import reload
import src.pipeline.saga as saga_mod; reload(saga_mod)
from src.pipeline.saga import query_2d_image

# 可通过自然语言查询2D场景中任意的物体特征 进行识别和分割
object = "shelf"

# 任意一张图像名（去 data/colmap_workspace/team_data2/dense/images/ 下看文件名）
query_2d_image("IMG_20260402_120500.jpg", object, saga_cfg)



Embedding dimension 512


#### 9.6.3  Step 3：估算 mask 3D 物理尺度（输出到 <image_root>/mask_scales/）


In [ ]:
# 训练前必须运行，约 3-5 分钟
compute_scales(saga_cfg)



[SAGA 9.6.3] 估算 mask 3D 物理尺度...
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/get_scale.py -m ...
Looking for config file in /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Config file found: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Loading trained model at iteration 30000, None
Allow Camera Principle Point Shift: False

Reading camera 1/117
Reading camera 2/117
Reading camera 3/117
Reading camera 4/117
Reading camera 5/117
Reading camera 6/117
Reading camera 7/117
Reading camera 8/117
Reading camera 9/117
Reading camera 10/117
Reading camera 11/117
Reading camera 12/117
Reading camera 13/117
Reading camera 14/117
Reading camera 15/117
Reading camera 16/117
Reading camera 17/117
Reading camera 18/117
Reading camera 19/117
Reading camera 20/117
Reading camera 21/117
Reading camera 22/117
Reading camera 23/117
Re

True

#### 9.6.4  Step 4：在冻结的 3DGS 上训练对比特征（~10-40 分钟）


In [ ]:
# 输出：<model_path>/point_cloud/.../contrastive_feature_point_cloud.ply
train_saga_features(saga_cfg)



[SAGA 9.6.4] 训练对比特征...
   模型路径：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   场景数据：/home/ansatz/github/ME6402-3D-Autonomous-Retail/data/colmap_workspace/team_data2/dense
   预计时间：10~40 分钟
   命令：/home/ansatz/miniconda3/envs/gaussian_splatting/bin/python /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/SAGA/train_contrastive_feature.py -m ...
Looking for config file in /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Config file found: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/cfg_args
Optimizing /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
RFN weight: 1.0 [05/04 23:04:43]
Smooth K: 16 [05/04 23:04:43]
Scale aware dim: -1 [05/04 23:04:43]
Loading trained model at iteration 30000, None [05/04 23:04:43]
Allow Camera Principle Point Shift: True [05/04 23:04:43]

Reading camera 1/117
Reading camera 2/117
Reading camera 3/117
R

True

### 9.7 SAGA 交互GUI分割和特征识别

#### 9.7.1  方式 A：启动 SAGA 交互 GUI（点击分割，推荐）

In [ ]:
# 操作：勾选 clickmode → 右键点击物体 → segment3d → save as
open_saga_gui(saga_cfg, gpu=0)



[SAGA GUI] 启动交互式分割界面...
   模型：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   场景 iter：30000，特征 iter：10000，GPU：0
   操作：勾选 clickmode → 右键点击物体 → segment3d → save as
✓ GUI 已启动，等待窗口弹出（约 10-20 秒）


loading model file...
project mat initialized !
loading model file done.
clickmode_multi_button =  True
[931.0, 407.0]
[985.0, 430.0]
[587.0, 351.0]
[591.0, 281.0]
[597.0, 388.0]


#### 9.7.2  文字查询 → 自动定位物体 → 3D Bounding Box


In [13]:
result = query_by_text("coca cola can", saga_cfg)
print(result)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 12025
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-07 19:34:01,678  INFO  query_by_text [coca cola can]: center=[-7.164441108703613, 0.4705319404602051, 5.008073806762695], size=[3.360989570617676, 1.9989196062088013, 1.727150797843933]
INFO:pipeline:query_by_text [coca cola can]: center=[-7.164441108703613, 0.4705319404602051, 5.008073806762695], size=[3.360989570617676, 1.9989196062088013, 1.727150797843933]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/coca_cola_can.pt  (选中 Gaussian 数：367)

✅ 3D Bounding Box — coca cola can
   中心坐标 : [-7.1644, 0.4705, 5.0081]  （单位：米）
   尺寸 XYZ : [3.3610, 1.9989, 1.7272]
   包含 Gaussian 数 : 367
{'label': 'coca cola can', 'center': [-7.164441108703613, 0.4705319404602051, 5.008073806762695], 'size': [3.360989570617676, 1.9989196062088013, 1.727150797843933], 'bbox_min': [-8.84493637084961, -0.5289278626441956, 4.144498348236084], 'bbox_max': [-5.483946323394775, 1.469991683959961, 5.871649265289307], 'n_gaussians': 367, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/coca_cola_can.pt'}


In [14]:
result2 = query_by_text("facial tissue box", saga_cfg, score_threshold=0.7)
print(result2)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 12100
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-07 19:34:41,678  INFO  query_by_text [facial tissue box]: center=[-2.377291679382324, 2.8916425704956055, -1.0666661262512207], size=[13.096017837524414, 11.922868728637695, 20.079191207885742]
INFO:pipeline:query_by_text [facial tissue box]: center=[-2.377291679382324, 2.8916425704956055, -1.0666661262512207], size=[13.096017837524414, 11.922868728637695, 20.079191207885742]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/facial_tissue_box.pt  (选中 Gaussian 数：851)

✅ 3D Bounding Box — facial tissue box
   中心坐标 : [-2.3773, 2.8916, -1.0667]  （单位：米）
   尺寸 XYZ : [13.0960, 11.9229, 20.0792]
   包含 Gaussian 数 : 851
{'label': 'facial tissue box', 'center': [-2.377291679382324, 2.8916425704956055, -1.0666661262512207], 'size': [13.096017837524414, 11.922868728637695, 20.079191207885742], 'bbox_min': [-8.925300598144531, -3.069791793823242, -11.10626220703125], 'bbox_max': [4.170717239379883, 8.853076934814453, 8.972929000854492], 'n_gaussians': 851, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/facial_tissue_box.pt'}


In [15]:
result3 = query_by_text("potato chips bag", saga_cfg)
print(result3)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 12124
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-07 19:35:19,180  INFO  query_by_text [potato chips bag]: center=[-1.7546932697296143, 2.956191301345825, -3.481905937194824], size=[10.349531173706055, 6.766422748565674, 10.186369895935059]
INFO:pipeline:query_by_text [potato chips bag]: center=[-1.7546932697296143, 2.956191301345825, -3.481905937194824], size=[10.349531173706055, 6.766422748565674, 10.186369895935059]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/potato_chips_bag.pt  (选中 Gaussian 数：3005)

✅ 3D Bounding Box — potato chips bag
   中心坐标 : [-1.7547, 2.9562, -3.4819]  （单位：米）
   尺寸 XYZ : [10.3495, 6.7664, 10.1864]
   包含 Gaussian 数 : 3,005
{'label': 'potato chips bag', 'center': [-1.7546932697296143, 2.956191301345825, -3.481905937194824], 'size': [10.349531173706055, 6.766422748565674, 10.186369895935059], 'bbox_min': [-6.9294586181640625, -0.4270200729370117, -8.575090408325195], 'bbox_max': [3.420072317123413, 6.339402675628662, 1.611279010772705], 'n_gaussians': 3005, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/potato_chips_bag.pt'}


In [16]:
result4 = query_by_text("juice drink pouch", saga_cfg)
print(result4)


   使用 contrastive feature iter=10000
Loading trained model at iteration 30000, 10000
Allow Camera Principle Point Shift: False
Reading camera 117/117
Loading Training Cameras
Loading Test Cameras
   anchor points: 11895
   训练相机数：117，开始收集 CLIP 特征...
   累计 mask 数：12500


2026-04-07 19:35:59,859  INFO  query_by_text [juice drink pouch]: center=[-7.028055191040039, 1.2885992527008057, 4.963006019592285], size=[3.4707603454589844, 3.786378860473633, 2.1192705631256104]
INFO:pipeline:query_by_text [juice drink pouch]: center=[-7.028055191040039, 1.2885992527008057, 4.963006019592285], size=[3.4707603454589844, 3.786378860473633, 2.1192705631256104]


Embedding dimension 512
✓ mask 已保存 → /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/juice_drink_pouch.pt  (选中 Gaussian 数：371)

✅ 3D Bounding Box — juice drink pouch
   中心坐标 : [-7.0281, 1.2886, 4.9630]  （单位：米）
   尺寸 XYZ : [3.4708, 3.7864, 2.1193]
   包含 Gaussian 数 : 371
{'label': 'juice drink pouch', 'center': [-7.028055191040039, 1.2885992527008057, 4.963006019592285], 'size': [3.4707603454589844, 3.786378860473633, 2.1192705631256104], 'bbox_min': [-8.763435363769531, -0.6045901775360107, 3.9033708572387695], 'bbox_max': [-5.292675018310547, 3.181788682937622, 6.022641181945801], 'n_gaussians': 371, 'mask_path': '/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/segmentation_res/juice_drink_pouch.pt'}


#### 9.7.3  渲染分割结果（render.py）

将分割出的物体渲染成图片，查看每个训练视角下的效果：
- `--target scene --segment`：渲染带分割的场景（背景移除）
- `--target seg`：渲染 2D 黑白 mask 图

In [23]:
from importlib import reload
import src.pipeline.saga as saga_mod; reload(saga_mod)
from src.pipeline.saga import visualize_query_results

visualize_query_results(result, result2, result3, result4)


#### 9.7.4  3D 识别框查看器（高斯泼溅真实感渲染 + YOLO 风格 3D 框）

独立查看器，与 SAGA GUI 互不干扰：
- **高斯泼溅真实感渲染**（和 SAGA GUI 同级画质）
- **黄色 3D 线框 + 标签**，旋转视角时框随透视变换
- 左键旋转 / 右键平移 / 滚轮缩放
- 支持同时显示多个物体识别框（传入 list）


In [ ]:
# 使用上一步 query_by_text 返回的 bbox 直接启动查看器
# 也可传入多个：open_bbox_viewer([bbox1, bbox2], saga_cfg, gpu=1)

bboxes = [result, result2, result3, result4]  # result 是之前 coca cola can 的结果
open_bbox_viewer(bboxes, saga_cfg)




[BBox Viewer] 启动 3D 识别框查看器...
   模型：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
   识别物体：['coca cola can', 'facial tissue box', 'potato chips bag', 'juice drink pouch']，GPU：1
   操作：左键旋转 / 右键平移 / 滚轮缩放
✓ 查看器已启动，等待窗口弹出（约 10-20 秒）


[BBoxViewer] Loading model: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter
[BBoxViewer] PLY: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/3dgs_team_data2_30000iter/point_cloud/iteration_30000/point_cloud.ply
[BBoxViewer] Loaded 1,201,772 Gaussians


In [ ]:
from importlib import reload
import src.pipeline.saga as saga_mod; reload(saga_mod)
from src.pipeline.saga import validate_3d_fusion

validate_3d_fusion("coca_cola_can", saga_cfg)


   'coca_cola_can'：选中 366 个 Gaussians，投影到 6 个视角


/home/ansatz/github/ME6402-3D-Autonomous-Retail/src/pipeline/saga.py:1445: UserWarning: Glyph 39564 (\N{CJK UNIFIED IDEOGRAPH-9A8C}) missing from font(s) DejaVu Sans.
/home/ansatz/github/ME6402-3D-Autonomous-Retail/src/pipeline/saga.py:1445: UserWarning: Glyph 35777 (\N{CJK UNIFIED IDEOGRAPH-8BC1}) missing from font(s) DejaVu Sans.


In [ ]:
for label in ["coca_cola_can", "facial_tissue_box", "potato_chips_bag", "juice_drink_pouch"]:
    validate_3d_fusion(label, saga_cfg, n_views=4)


   'coca_cola_can'：选中 367 个 Gaussians，投影到 4 个视角
   'facial_tissue_box'：选中 818 个 Gaussians，投影到 4 个视角
   'potato_chips_bag'：选中 4480 个 Gaussians，投影到 4 个视角
   'juice_drink_pouch'：选中 371 个 Gaussians，投影到 4 个视角
